# Titanic — your first supervised pipeline

This is a **fill-in-the-blanks project**, not a finished notebook. That's deliberate. You learn by deciding, not by running my code.

**Goal:** predict whether a passenger survived (a yes/no = *classification* problem).

### How this notebook works
- Cells marked **✅ GIVEN** are boilerplate I've done for you (the rote stuff — loading, plotting). Just run them.
- Cells marked **❓ YOUR TURN** have `# TODO` blanks for *you* to fill. Hints are provided. Answers are not.
- **The one rule:** try every YOUR TURN cell *yourself* before asking anyone (including AI) for the answer. Getting stuck and unstuck is the learning. If you're stuck for more than ~15 minutes, *then* ask — but ask "here's what I tried, here's the error," not "do it for me."

### Setup before you start
1. Download `train.csv` from kaggle.com/c/titanic and put it in the same folder as this notebook.
2. Run the setup cell below.


## §1 Setup ✅ GIVEN

In [3]:
import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Ready.")


Ready.


## §2 Load and inspect ✅ GIVEN

Same first-contact ritual as your clustering project: shape, types, missing values.


In [4]:
df = pd.read_csv('train.csv')
print("Shape:", df.shape)
df.head()


Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [11]:
# how to check each column's unique values
df.nunique()

PassengerId    891
Survived         2
Pclass           3
Name           891
Sex              2
Age             88
SibSp            7
Parch            7
Ticket         681
Fare           248
Cabin          147
Embarked         3
dtype: int64

In [6]:
# Missing values per column — pay close attention to this output.
print("Missing values per column:")
print(df.isnull().sum())


Missing values per column:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


**Read that missing-values output carefully.** Three columns have gaps:
- `Age` — about 20% missing. Too useful to drop the column; you'll need to *fill* it.
- `Cabin` — about 77% missing. Mostly empty. A column this empty is usually more trouble than it's worth.
- `Embarked` — only 2 missing. Trivial to fill.

How you handle each of these is YOUR decision in §4. First, let's understand the data.


## §3 Explore the data ✅ GIVEN (plots) / ❓ YOUR TURN (interpret)

Run these plots, then write down what you notice in the markdown cell after.


In [7]:
# Overall survival rate
rate = df['Survived'].mean()
print(f"Overall survival rate: {rate:.1%}")

# Survival by sex
px.histogram(df, x='Sex', color='Survived', barmode='group',
             title='Survival by sex').show()


Overall survival rate: 38.4%


In [8]:
# Survival by passenger class
px.histogram(df, x='Pclass', color='Survived', barmode='group',
             title='Survival by passenger class').show()

# Age distribution
px.histogram(df, x='Age', nbins=40, title='Age distribution').show()


### ❓ YOUR TURN — write what you see
Double-click this cell and answer (no code, just observations):

1. Did men or women survive at a higher rate? By roughly how much?
2. Which passenger class survived best? Worst?
3. What shape is the Age distribution? Roughly symmetric, or skewed?

> *Your answers here:*
> 1.women survived almost a 100 difference bettween the survival plots. percentage wise. men are just 18% survival , women are 74.20% roughly

> 2.class 3 died most. just with the numbers survived are kinda similar. class 1 is more

> 3.kinda skewed



## §4 Clean the data ❓ YOUR TURN — this is the core lesson

This is the part Country-data couldn't teach you, because it was already clean. Here you make real cleaning decisions. There's no single "correct" answer — there are reasonable choices with tradeoffs (the theme of this whole journey).

Fill in the `# TODO` lines below. Hints are in the comments.


In [12]:
df_clean = df.copy()

# --- TODO 1: Fill missing Age ---
# The simplest choice: fill with the median age.
# A smarter choice: fill with the median age *within each Pclass* (richer
# passengers skewed older). Pick one. Simple is fine for a first pass.
# Hint (simple):   df_clean['Age'] = df_clean['Age'].fillna(  ???  )
# Hint (median fn): df_clean['Age'].median()
# df_clean['Age'] = df_clean['Age'].fillna(0)   # <-- replace the 0 with your choice
df_clean['Age'] = df_clean['Age'].fillna(
    df_clean.groupby('Pclass')['Age'].transform('median')
)


# --- TODO 2: Handle Cabin (77% missing) ---
# Option A: drop the column entirely (it's mostly empty).
# Option B: turn it into a flag: 1 if a cabin was recorded, 0 if not
#           (maybe having a recorded cabin signals something).
# Hint (drop):  df_clean = df_clean.drop(columns=[ ??? ])
# Hint (flag):  df_clean['has_cabin'] = df_clean['Cabin'].notna().astype(int)
#               ...then drop the original 'Cabin'.
# TODO: write your choice here
df_clean['has_cabin'] = df_clean['Cabin'].notna().astype(int)
df_clean = df_clean.drop(columns=['Cabin'])


# --- TODO 3: Fill the 2 missing Embarked values ---
# Easiest: fill with the most common value (the mode).
# Hint: df_clean['Embarked'].mode()[0]  gives the most frequent port.
# TODO: df_clean['Embarked'] = df_clean['Embarked'].fillna( ??? )
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0]) # [0] means the first mode value, similarly [1] would be the second most common value


# Check your work — this should print all zeros (or near it) when you're done:
print("Missing values after cleaning:")
print(df_clean.isnull().sum())


Missing values after cleaning:
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
has_cabin      0
dtype: int64


## §5 Engineer features ❓ YOUR TURN

Models need numbers. Right now `Sex` and `Embarked` are text, and `Name`/`Ticket` are messy. Turn them into something usable. I've done the first one as an example — you do the rest.


In [13]:
# ✅ EXAMPLE (given): encode Sex as a number
df_clean['Sex'] = df_clean['Sex'].map({'male': 0, 'female': 1})


# --- TODO 4: Encode Embarked ---
# It has 3 values (C, Q, S). Best practice = one-hot encode (one column per port).
# Hint: pd.get_dummies(df_clean, columns=['Embarked'], drop_first=True)
#       (reassign the result back to df_clean)
# TODO:
df_clean = pd.get_dummies(df_clean, columns=['Embarked'], drop_first=True) # it will create columns for each unique value in Embarked, except the first one


# --- TODO 5 (optional but powerful): extract Title from Name ---
# Names look like "Braund, Mr. Owen Harris". The title ('Mr', 'Mrs', 'Miss',
# 'Master') is surprisingly predictive of survival.
# Hint: df_clean['Title'] = df_clean['Name'].str.extract(r' ([A-Za-z]+)\.')
#       then encode it (rare titles can be grouped into 'Other'),
#       then one-hot or map it to numbers.
# TODO (optional):
df_clean['Title'] = df_clean['Name'].str.extract(r' ([A-Za-z]+)\.')
df_clean['Title'] = df_clean['Title'].replace(['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Other')
df_clean['Title'] = df_clean['Title'].map({'Mr': 1, 'Miss': 2, 'Mrs': 3, 'Master': 4, 'Other': 5})
df_clean = pd.get_dummies(df_clean, columns=['Title'], drop_first=True)



# --- TODO 6: drop columns the model can't use ---
# Name, Ticket, PassengerId (and Cabin if you didn't already) are text/IDs
# the model can't learn from. Drop them.
# Hint: df_clean = df_clean.drop(columns=[ ??? ])
# TODO:
df_clean = df_clean.drop(columns=['Name', 'Ticket', 'PassengerId'])

print("Columns remaining (should all be numeric now):")
print(df_clean.dtypes)


Columns remaining (should all be numeric now):
Survived        int64
Pclass          int64
Sex             int64
Age           float64
SibSp           int64
Parch           int64
Fare          float64
has_cabin       int64
Embarked_Q       bool
Embarked_S       bool
Title_2.0        bool
Title_3.0        bool
Title_4.0        bool
Title_5.0        bool
dtype: object


## §6 Build the model

First model is GIVEN so you see the pattern. The second model is YOUR TURN.


In [14]:
# ✅ GIVEN — set up X (features) and y (target), then split into train/test.
y = df_clean['Survived']
X = df_clean.drop(columns=['Survived'])

# 80% to train on, 20% held back to test honestly.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"Training rows: {len(X_train)}, Test rows: {len(X_test)}")


Training rows: 712, Test rows: 179


In [ ]:
# ✅ GIVEN — first model: Logistic Regression
# (the natural model for yes/no outcomes — remember the Binomial story).
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)
pred_lr = logreg.predict(X_test)

print(f"Logistic Regression accuracy: {accuracy_score(y_test, pred_lr):.3f}")


### ❓ YOUR TURN — try a second model and compare
Fill in a `RandomForestClassifier` (or `DecisionTreeClassifier`) below, the same way the logistic regression was done. Then see which scores higher.


In [ ]:
# --- TODO 7: train a second model ---
# Hint: it's the same 3 lines as above, swapping the model:
#   model = RandomForestClassifier(n_estimators=100, random_state=42)
#   model.fit(X_train, y_train)
#   preds = model.predict(X_test)
#   print accuracy_score(y_test, preds)
# TODO:


## §7 Evaluate ✅ GIVEN (run for your best model)

Accuracy is one number. The confusion matrix shows *what kind* of mistakes the model makes (did it miss survivors, or falsely predict survival?).


In [ ]:
# Change `pred_lr` to your best model's predictions if the second model won.
best_preds = pred_lr

print("Confusion matrix (rows = actual, cols = predicted):")
print(confusion_matrix(y_test, best_preds))
print()
print(classification_report(y_test, best_preds))


## §8 Reflect ❓ YOUR TURN — the lesson

Double-click and answer in words. This is where the understanding locks in.

1. **Which model won, and by how much?**
2. **Did you use PCA in this project? Should you have?**
   *(Think back: how many features do you have — a handful, or 80 like House Prices? What did PCA actually buy you in the country project? This is the "when NOT to use a tool" judgment.)*
3. **What was the hardest cleaning decision, and what did you choose?**
4. **If you got a higher score, what's one thing you'd try next?**

> *Your answers here:*
>
> 1.
> 2.
> 3.
> 4.


## §9 You're done — what's next

If you filled every TODO and got a model scoring (most people land around 78–82% accuracy), **you have completed a full supervised ML pipeline end to end**: load → explore → clean → engineer features → train → evaluate. That's the whole shape. You've now done it for *both* unsupervised (clustering) and supervised (classification) problems.

**Next project: House Prices** (kaggle.com/c/house-prices-advanced-regression-techniques). It's the graduation:
- 80 columns → PCA and feature selection finally *matter*
- Messy missing values with different meanings → real cleaning
- A skewed target → your log-transform skill gets used
- It's *regression* (predict a number), completing your tour of the two main problem types

Bring your finished Titanic notebook back and we'll start House Prices together — and you'll feel the difference between "I ran AI's code" and "I built this."
